# Etapa 5 — Grad-CAM e exemplos de acertos e erros

O Grad-CAM destaca as regiões que contribuem para a saída da classe ALTERADA. A inspeção é qualitativa e não constitui evidência de raciocínio clínico.

Use o checkpoint treinado na Etapa 2 e as imagens geradas pela Etapa 4 neste checkout. No Colab, o notebook também procura o checkpoint na pasta do projeto no Drive.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

in_colab = "google.colab" in sys.modules
if Path.cwd().name == "notebooks":
    os.chdir("..")
if in_colab and not Path("src/config.py").is_file():
    repo = Path("triagem-raio-x")
    if not repo.is_dir():
        subprocess.run(
            ["git", "clone", "https://github.com/LuizCenci/triagem-raio-x", str(repo)],
            check=True,
        )
    os.chdir(repo)
if in_colab:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "kagglehub", "opencv-python-headless"],
        check=True,
    )
sys.path.insert(0, str(Path.cwd().resolve()))

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from src.config import CLASSES, DATA_DIR, RESULTS_DIR, ROOT, SPLITS_DIR
from src.decision import Thresholds, decide
from src.degrade import _read_gray
from src.download import download_dataset
from src.metadata import attach_paths
from src.quality import is_inadequate, quality_features

figure_dir = RESULTS_DIR / "figures"
figure_dir.mkdir(parents=True, exist_ok=True)
th = Thresholds.from_json(RESULTS_DIR / "thresholds.json")
limits = json.loads((RESULTS_DIR / "quality_limits.json").read_text(encoding="utf-8"))
test = pd.read_csv(RESULTS_DIR / "predictions" / "resnet50_test.csv")
cases = ("VP", "FN", "FP", "VN")


def classify(frame):
    positive = frame["p_alterada"] >= th.t_rotina
    altered = frame["label"].eq(1)
    return np.select(
        [altered & positive, altered & ~positive, ~altered & positive],
        cases[:3],
        default=cases[3],
    )


def rank(frame, column, ascending):
    return frame.sort_values([column, "image"], ascending=[ascending, True])


test["caso"] = classify(test)
selected_test = pd.concat([
    rank(test.loc[test["caso"] == case], "p_alterada", ascending=case in ("FN", "VN")).head(3)
    for case in cases
], ignore_index=True)
if (selected_test["caso"].value_counts().reindex(cases, fill_value=0) != 3).any():
    raise ValueError("São necessários três casos de cada tipo (VP, FN, FP e VN) para a grade.")
display(selected_test[["image", "caso", "label", "p_alterada"]])

## 1. Modelo e arquivos de imagem

In [ ]:
import keras
import tensorflow as tf

from src.pipeline import IMG_SIZE, load_image, make_dataset

model_path = ROOT / "models" / "resnet50.keras"
if in_colab and not model_path.is_file():
    from google.colab import drive

    drive.mount("/content/drive")
    model_path = Path("/content/drive/MyDrive/triagem-raio-x/models/resnet50.keras")
if not model_path.is_file():
    raise FileNotFoundError(f"Checkpoint treinado da Etapa 2 ausente em {model_path}.")
model = keras.models.load_model(model_path, compile=False)
preprocess_input = keras.applications.resnet50.preprocess_input

raw_dir = DATA_DIR / "raw" / "images"
images_root = raw_dir if raw_dir.is_dir() else download_dataset()
split = pd.read_csv(SPLITS_DIR / "test.csv", usecols=["image", "label"])
originals = attach_paths(split, images_root)
test = test.merge(originals, on=["image", "label"], how="left", validate="one_to_one")
if test["path"].isna().any():
    raise FileNotFoundError("Há imagens do teste sem arquivo original ou com rótulo divergente.")
selected_test = selected_test.merge(test[["image", "path"]], on="image", validate="one_to_one")

degraded_dir = DATA_DIR / "degraded"
metadata_path = degraded_dir / "degraded_metadata.csv"
if not metadata_path.is_file():
    raise FileNotFoundError("Execute a geração de imagens do notebook 04_degradado.ipynb.")
degraded = pd.read_csv(RESULTS_DIR / "degraded_metadata.csv")
if not pd.read_csv(metadata_path).equals(degraded):
    raise ValueError("Os metadados locais divergem dos versionados. Regenere as imagens na Etapa 4.")
degraded["path"] = degraded["image"].map(lambda name: str(degraded_dir / name))
if not degraded["path"].map(lambda path: Path(path).is_file()).all():
    raise FileNotFoundError("Há imagens degradadas ausentes. Regenere as imagens na Etapa 4.")
pairs = test[["image", "label", "path", "p_alterada"]].rename(columns={
    "image": "original_image", "path": "original_path", "p_alterada": "p_original",
})
degraded = degraded.merge(pairs, on=["original_image", "label"], how="left", validate="one_to_one")
if degraded["original_path"].isna().any():
    raise ValueError("Há metadados degradados sem predição original correspondente no teste.")

## 2. Grad-CAM e conferência da probabilidade

In [ ]:
def grad_cam_and_probability(model, img_224x224x3):
    """Devolve o mapa da classe ALTERADA e a probabilidade usada no gradiente."""
    base = model.get_layer("resnet50")
    target = base.get_layer("conv5_block3_out")
    grad_model = keras.Model(base.inputs, target.output)
    batch = preprocess_input(tf.expand_dims(img_224x224x3, axis=0))
    conv = grad_model(batch, training=False)

    with tf.GradientTape() as tape:
        tape.watch(conv)
        pooled = model.get_layer("gap")(conv)
        pred = model.get_layer("head")(pooled)[:, 0]
    grads = tape.gradient(pred, conv)
    if grads is None:
        raise RuntimeError("Sem gradiente da saída ALTERADA para conv5_block3_out.")

    weights = tf.reduce_mean(grads[0], axis=(0, 1))
    cam = tf.nn.relu(tf.reduce_sum(conv[0] * weights, axis=-1)).numpy()
    cam /= max(float(cam.max()), 1e-8)
    cam = cv2.resize(cam, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
    return np.clip(cam, 0, 1), float(pred[0])


def grad_cam(model, img_224x224x3) -> np.ndarray:
    return grad_cam_and_probability(model, img_224x224x3)[0]


sanity_case = selected_test.iloc[0]
_, calculated = grad_cam_and_probability(model, load_image(sanity_case["path"]))
difference = abs(calculated - sanity_case["p_alterada"])
if not difference < 1e-4:
    raise ValueError(
        f"Predição divergente em {sanity_case['image']}: "
        f"checkpoint={calculated:.8f}, CSV={sanity_case['p_alterada']:.8f}, diferença={difference:.2g}."
    )
print(f"{sanity_case['image']}: diferença entre o checkpoint e o CSV = {difference:.2g}")

## 3. Grad-CAM no teste e no conjunto degradado

In [ ]:
def overlay(path):
    gray = _read_gray(path)
    heatmap = grad_cam(model, load_image(path))
    original = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
    if not heatmap.any():
        return original
    height, width = gray.shape
    heatmap = cv2.resize(heatmap, (width, height), interpolation=cv2.INTER_LINEAR)
    color = cv2.applyColorMap(np.rint(255 * heatmap).astype(np.uint8), cv2.COLORMAP_JET)
    return cv2.addWeighted(original, 0.6, cv2.cvtColor(color, cv2.COLOR_BGR2RGB), 0.4, 0)


def save_figure(fig, filename):
    fig.savefig(figure_dir / filename, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)


fig, axes = plt.subplots(4, 3, figsize=(11, 14), layout="constrained")
for ax, row in zip(axes.flat, selected_test.itertuples()):
    ax.imshow(overlay(row.path))
    ax.set_title(f"{row.caso} · {CLASSES[row.label]} · p={row.p_alterada:.3f}\n{row.image}")
    ax.axis("off")
save_figure(fig, "gradcam_teste.png")

In [ ]:
dataset = make_dataset(degraded, preprocess_input, shuffle=False)
degraded["p_alterada"] = model.predict(dataset, verbose=0).ravel()
degraded["caso"] = classify(degraded)
degraded["queda"] = degraded["p_original"] - degraded["p_alterada"]
selected_degraded = rank(degraded.loc[degraded["label"].eq(1)], "queda", ascending=False).head(3)
if len(selected_degraded) != 3:
    raise ValueError("São necessárias três imagens ALTERADAS no conjunto degradado.")

fig, axes = plt.subplots(3, 2, figsize=(8, 11), layout="constrained")
for (left, right), row in zip(axes, selected_degraded.itertuples()):
    left.imshow(overlay(row.original_path))
    left.set_title(f"Original · ALTERADA · p={row.p_original:.3f}\n{row.original_image}")
    right.imshow(overlay(row.path))
    kind = row.degradation.replace("_", " ")
    right.set_title(f"{kind} · {row.caso} · p={row.p_alterada:.3f}\nQueda de p = {row.queda:+.3f}")
    for ax in (left, right):
        ax.axis("off")
save_figure(fig, "gradcam_degradado.png")

## 4. Exemplos de acertos e erros

Acertos e erros usam o limiar de rotina. A saída de prioridade também considera a checagem de qualidade. Dentro de cada grupo, a seleção prioriza a distância ao limiar e desempata pelo nome do arquivo.

In [ ]:
def select_examples(frame, correct, count):
    matches = frame["caso"].isin(("VP", "VN"))
    candidates = frame.loc[matches == correct].copy()
    candidates["distancia"] = (candidates["p_alterada"] - th.t_rotina).abs()
    return rank(candidates, "distancia", ascending=False).head(count)


groups = [
    ("Teste · acertos", select_examples(test, True, 4), 4),
    ("Teste · erros", select_examples(test, False, 4), 4),
    ("Degradado · acertos", select_examples(degraded, True, 2), 2),
    ("Degradado · erros", select_examples(degraded, False, 2), 2),
]
fig, axes = plt.subplots(4, 4, figsize=(15, 14), layout="constrained")
for row_axes, (name, examples, count) in zip(axes, groups):
    for ax in row_axes:
        ax.axis("off")
    for ax, row in zip(row_axes, examples.itertuples()):
        gray = _read_gray(row.path)
        inadequate = is_inadequate(quality_features(gray), limits)
        outcome = decide(row.p_alterada, th, inadequate=inadequate)
        ax.imshow(gray, cmap="gray", vmin=0, vmax=255)
        ax.set_title(
            f"{name} · {row.caso}\n{CLASSES[row.label]} · p={row.p_alterada:.3f}\n{outcome}",
            fontsize=9,
        )
    if len(examples) < count:
        row_axes[len(examples)].text(
            0.5, 0.5, f"{name}\n{len(examples)} de {count} disponíveis no limiar",
            ha="center", va="center",
        )
save_figure(fig, "acertos_erros.png")

## 5. Inspeção visual dos 15 casos

Examine as duas grades e preencha as colunas `foco` e `observacao` em `data/gradcam_review.csv`, depois rode novamente a célula abaixo. Use `pulmao`, `fora` ou `misto` e descreva o que vê em cada mapa, dando atenção especial aos falsos negativos. Evite atribuir significado clínico a bordas, textos, marcadores ou dispositivos.

In [ ]:
review_path = DATA_DIR / "gradcam_review.csv"
columns = ["image", "conjunto", "caso", "label", "p_alterada", "foco", "observacao"]
keys = ["image", "conjunto", "caso"]
notes = ["foco", "observacao"]
review = pd.concat([
    selected_test.assign(conjunto="teste"),
    selected_degraded.assign(conjunto="degradado"),
], ignore_index=True)[columns[:-2]]

if review_path.is_file():
    previous = pd.read_csv(review_path, keep_default_na=False)
    review = review.merge(previous[keys + notes], on=keys, how="left", validate="one_to_one")
    review[notes] = review[notes].fillna("")
else:
    review[notes] = ""
review[notes] = review[notes].apply(lambda column: column.str.strip())
review.to_csv(review_path, index=False, encoding="utf-8")

complete = review["foco"].isin(("pulmao", "fora", "misto")) & review["observacao"].ne("")
if not complete.all():
    print(f"Inspeção pendente em {(~complete).sum()} de {len(review)} casos. Edite {review_path}.")
    display(review.loc[~complete, keys])
else:
    review.to_csv(RESULTS_DIR / "gradcam_casos.csv", index=False, encoding="utf-8")
    display(pd.crosstab([review["conjunto"], review["caso"]], review["foco"]))
    summary = ["### Observações dos falsos negativos"]
    for row in review.loc[review["caso"].eq("FN")].itertuples():
        summary.append(f"- **{row.image}** ({row.conjunto}, {row.foco}) — {row.observacao}")
    display(Markdown("\n".join(summary)))

## 6. Síntese da inspeção visual

Nos três falsos negativos do teste, os focos mais intensos aparecem fora do campo pulmonar, nas bordas superior ou inferior da imagem. O caso degradado classificado como falso negativo também destaca a borda superior, e sua probabilidade de alteração diminui após a redução de contraste. Esses mapas não permitem determinar quais achados o modelo deixou de reconhecer.

Nos verdadeiros positivos do teste, dois mapas concentram a ativação nos pulmões e um mistura região pulmonar e borda. Entre os falsos positivos, dois focam regiões pulmonares e um inclui a borda inferior. Nos verdadeiros negativos selecionados, a ativação mais intensa aparece fora dos pulmões. Os outros dois casos degradados mantêm a classificação de verdadeiro positivo, apesar da queda de probabilidade.

A classificação de foco é uma inspeção visual qualitativa dos mapas e não demonstra causalidade nem validade clínica. Marcadores, dispositivos e bordas presentes nas radiografias podem funcionar como atalhos visuais e merecem análise adicional.